# LLMInspector — Getting Started

A guided tour of the package: **schema → model → metrics → evaluate → synthesize → report**.

The offline cells (schema, adversarial synthesis, reporting shape) run without credentials. The
cells that call an LLM or ragas need a live Azure OpenAI model and the model dependencies.

## 1. Test cases & datasets (schema layer)

In [ ]:
from llminspector.dataset import EvaluationDataset
from llminspector.test_case import LLMTestCase

dataset = EvaluationDataset(test_cases=[
    LLMTestCase(
        input="What is the capital of France?",
        actual_output="Paris is the capital of France.",
        expected_output="Paris",
        retrieval_context=["The capital of France is Paris."],
    ),
])
# Load from a spreadsheet instead:  EvaluationDataset.from_excel("data.xlsx")
dataset.to_pandas()

## 2. Configure a model (provider layer)

`AzureSettings` replaces the old `config.ini`. Provide exactly one credential: `api_key` **or**
`azure_ad_token_provider`.

In [ ]:
from llminspector.config import AzureSettings
from llminspector.models import AzureOpenAIModel

# Reads LLMINSPECTOR_AZURE_ENDPOINT / _API_VERSION / _API_KEY (and the model names).
settings = AzureSettings.from_env()
model = AzureOpenAIModel(settings)      # or AzureOpenAIModel(settings, azure_ad_token_provider=fn)

## 3. Metrics + 4. Evaluate

Metrics are objects built with the model. `evaluate()` drives them per row, skipping any whose
inputs are missing, and runs exactly the metrics you hand it.

`answer_correctness` is a single unified judge: it weighs ground-truth agreement, faithfulness to
the retrieval context, and relevancy to the question **in one prompt**, so content that is missing
from the ground truth but supported by the context and relevant to the question is not penalised.
It also emits its three sub-judgements as `answer_correctness_gt_agreement` / `_faithfulness` /
`_relevancy`. `retrieval_context` is optional — rows without one are judged on two factors instead
of three.

In [ ]:
from llminspector import a_evaluate, reporting
from llminspector.metrics import (
    AnswerCorrectnessMetric,
    BertScoreMetric,
    FaithfulnessMetric,
    SentimentMetric,
)

metrics = [
    FaithfulnessMetric(model),
    AnswerCorrectnessMetric(model),
    SentimentMetric(model, target="actual_output"),
    BertScoreMetric(),
]
# Jupyter already runs an event loop, so await the async entry point here.
# The sync `evaluate(...)` is for scripts.
result = await a_evaluate(dataset, metrics)
result.to_pandas()

## 5. Synthesize test data

The adversarial synthesizer runs offline. Alignment (HF-T5) and RAG (ragas) need extra deps.
Every synthesizer returns an `EvaluationDataset` of `Golden`s; extra columns live in
`Golden.metadata`.

In [ ]:
from llminspector.synthesizer import AdversarialSynthesizer

synth = AdversarialSynthesizer.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx", capability="all", sample_size=5,
)
synth.generate()
print("metadata columns:", synth.metadata_keys)
synth.to_pandas().head()

## 6. Report

The result serializes itself (`result.to_pandas()` / `result.to_excel()`). `reporting` adds the
analysis on top: `summary()` for per-metric numeric stats, `errors()` for anything that failed.

In [ ]:
print(reporting.summary(result))

# If any metric failed, the table's Nones are ambiguous — check here:
reporting.errors(result)